# 🏠 Scraping - ZAP Imóveis
## Apartamentos à Venda em João Pessoa - PB

Este notebook implementa a extração de dados de anúncios imobiliários do portal **ZAP Imóveis** para apartamentos à **venda** em **João Pessoa - PB**.

### 💡 Compatibilidade Total com Jupyter + Windows + Python 3.14
- O Jupyter Notebook roda em um loop de eventos `asyncio` interno do Tornado.
- Para evitar o erro `It looks like you are using Playwright Sync API inside the asyncio loop` e o `NotImplementedError` do Windows, todas as chamadas do Playwright são executadas dentro de uma **thread OS dedicada** (`ThreadPoolExecutor`), onde o loop de eventos do Jupyter não interfere.
- **Seletores CSS Validados:**
  - **Preço, Condomínio e IPTU:** `p.value-item__value`
  - **Endereço e Bairro:** `[data-testid='location-address']`
  - **Características:** `span.amenities-item-text`
  - **Descrição:** `p.description__content--text`
- **Saída:** `imoveis_joao_pessoa_zap.json` com salvamento incremental

## 📦 Célula 1 — Imports e Isolador de Thread para Jupyter

In [1]:
import re
import os
import json
import time
import random
import sys
import asyncio
import concurrent.futures
from bs4 import BeautifulSoup
from playwright.sync_api import sync_playwright

def executar_em_thread(func, *args, **kwargs):
    """
    Executa qualquer função síncrona do Playwright em uma thread OS isolada.
    Garante o WindowsProactorEventLoopPolicy no Windows para evitar o erro NotImplementedError.
    """
    def wrapper():
        if sys.platform == 'win32':
            asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
            loop = asyncio.new_event_loop()
            asyncio.set_event_loop(loop)
        return func(*args, **kwargs)

    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as executor:
        future = executor.submit(wrapper)
        return future.result()

print('[OK] Imports e Isolador de Thread (ThreadPoolExecutor) prontos!')

✅ Imports e Isolador de Thread (ThreadPoolExecutor) prontos!


## ⚙️ Célula 2 — Configurações Globais e Anti-Bot Headers

In [2]:
BASE_LISTAGEM_URL = "https://www.zapimoveis.com.br/venda/apartamentos/pb+joao-pessoa/"
ARQUIVO_SAIDA = "imoveis_joao_pessoa_zap.json"
TAMANHO_LOTE = 50

USER_AGENT = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36'

EXTRA_HEADERS = {
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8',
    'Accept-Language': 'pt-BR,pt;q=0.9,en-US;q=0.8,en;q=0.7',
    'Sec-Ch-Ua': '"Chromium";v="122", "Not(A:Brand";v="24", "Google Chrome";v="122"',
    'Sec-Ch-Ua-Mobile': '?0',
    'Sec-Ch-Ua-Platform': '"Windows"',
    'Sec-Fetch-Dest': 'document',
    'Sec-Fetch-Mode': 'navigate',
    'Sec-Fetch-Site': 'same-origin',
    'Sec-Fetch-User': '?1',
    'Upgrade-Insecure-Requests': '1',
}

print(f"✅ Configurações e Cabeçalhos Stealth Prontos!")
print(f"   URL Base : {BASE_LISTAGEM_URL}")
print(f"   Saída    : {ARQUIVO_SAIDA}")

✅ Configurações e Cabeçalhos Stealth Prontos!
   URL Base : https://www.zapimoveis.com.br/venda/apartamentos/pb+joao-pessoa/
   Saída    : imoveis_joao_pessoa_zap.json


## 🔗 Célula 3 — Classe Principal: ScraperZapImoveis

In [3]:
class ScraperZapImoveis:
    def __init__(self):
        self.base_listagem_url = BASE_LISTAGEM_URL
        self.arquivo_saida = ARQUIVO_SAIDA
        self.tamanho_lote = TAMANHO_LOTE
        self.pagina = None
        self.url_atual = None
        self.soup = None

    @staticmethod
    def _normalizar_chave(texto):
        chave = texto.lower().strip()
        chave = re.sub(r'[áàâã]', 'a', chave)
        chave = re.sub(r'[éèê]',  'e', chave)
        chave = re.sub(r'[íìî]',  'i', chave)
        chave = re.sub(r'[óòôõ]', 'o', chave)
        chave = re.sub(r'[úùû]',  'u', chave)
        chave = re.sub(r'[ç]',    'c', chave)
        chave = re.sub(r'[\s\-]+', '_', chave)
        chave = re.sub(r'[^a-z0-9_]', '', chave)
        return chave.strip('_')

    # =========================================================================
    # ETAPA 1: COLETA DE LINKS VIA PAGINAÇÃO
    # =========================================================================

    def _detectar_total_paginas(self):
        print("🔍 Detectando total de páginas...")
        try:
            self.pagina.goto(self.base_listagem_url + "?pagina=1", wait_until="domcontentloaded", timeout=45000)
            time.sleep(2)
            html = self.pagina.content()
            soup = BeautifulSoup(html, 'html.parser')

            for tag in soup.find_all(['p', 'span', 'h1', 'h2']):
                texto = tag.get_text(strip=True)
                match = re.search(r'([\d\.]+)\s+im[oó]veis?', texto, re.IGNORECASE)
                if match:
                    total = int(match.group(1).replace('.', ''))
                    paginas = min((total // 24) + 1, 100)
                    print(f"   Total encontrado: {total} imóveis → ~{paginas} páginas")
                    return paginas

            nums = [int(m.group(1)) for a in soup.find_all('a', href=True) if (m := re.search(r'pagina=(\d+)', a.get('href', '')))]
            if nums:
                ultima = max(nums)
                print(f"   Última página identificada: {ultima}")
                return ultima
        except Exception as e:
            print(f"   ⚠️ Erro ao detectar páginas: {e}")

        return 100

    def _extrair_links_da_pagina(self, numero_pagina):
        url = f"{self.base_listagem_url}?pagina={numero_pagina}"
        try:
            self.pagina.goto(url, wait_until="domcontentloaded", timeout=45000)
            time.sleep(1.5)
            html = self.pagina.content()
            soup = BeautifulSoup(html, 'html.parser')

            links = []
            for a in soup.find_all('a', href=True):
                href = a['href']
                if '/imovel/' in href and 'joao-pessoa' in href.lower():
                    url_completa = href if href.startswith('http') else f"https://www.zapimoveis.com.br{href}"
                    links.append(url_completa)

            return list(dict.fromkeys(links))
        except Exception as e:
            print(f"   ❌ Erro ao ler página {numero_pagina}: {e}")
            return []

    def get_links_apartamentos_venda_jp(self):
        print("🚀 Coletando links de imóveis...\n")
        total_paginas = self._detectar_total_paginas()
        todos_os_links = []
        vazias_consecutivas = 0

        for num_pag in range(1, total_paginas + 1):
            print(f"  Página {num_pag}/{total_paginas}...", end=" ", flush=True)
            links_pag = self._extrair_links_da_pagina(num_pag)

            if not links_pag:
                vazias_consecutivas += 1
                print(f"(sem links) [{vazias_consecutivas}/3]")
                if vazias_consecutivas >= 3:
                    print("\n⛔ Fim dos resultados atingido.")
                    break
            else:
                vazias_consecutivas = 0
                todos_os_links.extend(links_pag)
                print(f"✔ {len(links_pag)} links (Total: {len(todos_os_links)})")

            time.sleep(random.uniform(1.2, 2.2))

        links_unicos = list(dict.fromkeys(todos_os_links))
        print(f"\n✅ Total de links únicos garantidos: {len(links_unicos)}")
        return links_unicos

    # =========================================================================
    # ETAPA 2: CARREGAMENTO DA PÁGINA DO ANÚNCIO
    # =========================================================================

    def _carregar_html_da_pagina(self):
        try:
            time.sleep(random.uniform(1.0, 2.0))
            self.pagina.goto(self.url_atual, wait_until="domcontentloaded", timeout=40000)
            time.sleep(1.5)
            html = self.pagina.content()
            self.soup = BeautifulSoup(html, 'html.parser')
            return True
        except Exception as e:
            print(f"  ❌ Erro ao carregar {self.url_atual}: {e}")
            self.soup = None
            return False

    # =========================================================================
    # ETAPA 3: EXTRAÇÃO DE DADOS (SELETORES VALIDADOS)
    # =========================================================================

    def _extrair_preco_e_custos(self, dados):
        elementos_valor = self.soup.select("p.value-item__value")
        if elementos_valor:
            venda_txt = elementos_valor[0].get_text(strip=True)
            dados['preco_venda'] = re.sub(r'R\$\s*', '', venda_txt).strip()
            
            for item in elementos_valor[1:]:
                txt = item.get_text(strip=True)
                parent = item.find_parent()
                parent_txt = parent.get_text(strip=True).lower() if parent else ""
                if 'condom' in parent_txt and 'condominio' not in dados:
                    dados['condominio'] = re.sub(r'R\$\s*', '', txt).replace('/mês', '').strip()
                elif 'iptu' in parent_txt and 'iptu' not in dados:
                    dados['iptu'] = re.sub(r'R\$\s*', '', txt).strip()
        else:
            tag_preco = self.soup.select_one("[data-testid='price-info-value']") or self.soup.select_one("[class*='price'] b")
            dados['preco_venda'] = re.sub(r'R\$\s*', '', tag_preco.get_text(strip=True)).strip() if tag_preco else None

        if 'condominio' not in dados:
            match_cond = re.search(r'Condom[íi]nio\s*R\$\s*([\d\.]+)', self.soup.get_text(), re.I)
            dados['condominio'] = match_cond.group(1) if match_cond else None
        if 'iptu' not in dados:
            match_iptu = re.search(r'IPTU\s*R\$\s*([\d\.]+)', self.soup.get_text(), re.I)
            dados['iptu'] = match_iptu.group(1) if match_iptu else None

    def _extrair_endereco_e_bairro(self, dados):
        tag_addr = self.soup.select_one("[data-testid='location-address']") or self.soup.find('address')
        if tag_addr:
            end_texto = tag_addr.get_text(strip=True)
            dados['endereco_completo'] = end_texto
            
            match_bairro = re.search(r'-\s*([^,]+),\s*Jo[aã]o Pessoa', end_texto, re.I)
            if match_bairro:
                dados['bairro'] = match_bairro.group(1).strip()
            else:
                partes = end_texto.split('-')
                dados['bairro'] = partes[-1].replace('João Pessoa', '').replace('PB', '').strip(', ') if len(partes) > 1 else None
        else:
            dados['endereco_completo'] = None
            dados['bairro'] = None

    def _extrair_caracteristicas(self, dados):
        items_amenities = self.soup.select("span.amenities-item-text")
        for item in items_amenities:
            txt = item.get_text(strip=True)
            txt_norm = txt.lower()
            
            if 'quarto' in txt_norm:
                dados['quartos'] = re.sub(r'[^\d]', '', txt)
            elif 'banheiro' in txt_norm:
                dados['banheiros'] = re.sub(r'[^\d]', '', txt)
            elif 'vaga' in txt_norm or 'garagem' in txt_norm:
                dados['vagas'] = re.sub(r'[^\d]', '', txt)
            elif 'm2' in txt_norm or 'm²' in txt:
                dados['area_util'] = re.sub(r'[^\d,\.]', '', txt).strip()

        h1_text = dados.get('titulo', '') or ''
        if 'quartos' not in dados:
            m_q = re.search(r'(\d+)\s*Quartos?', h1_text, re.I)
            if m_q: dados['quartos'] = m_q.group(1)
        if 'area_util' not in dados:
            m_a = re.search(r'(\d+)\s*m²', h1_text, re.I)
            if m_a: dados['area_util'] = m_a.group(1)

    def _extrair_detalhes_adicionais(self, dados):
        cod_match = re.search(r'id-(\d+)', self.url_atual or '')
        if cod_match:
            dados['codigo_imovel'] = cod_match.group(1)
        else:
            cod_txt = self.soup.find(string=re.compile(r'C[oó]d\.?\s*no\s*Zap', re.I))
            dados['codigo_imovel'] = re.search(r'\d{4,}', cod_txt).group(0) if cod_txt and re.search(r'\d{4,}', cod_txt) else None

        anunc_tag = self.soup.select_one("[data-testid='advertiser-name']") or self.soup.select_one("[class*='advertiser']")
        dados['anunciante'] = anunc_tag.get_text(strip=True) if anunc_tag else None

    def _extrair_comodidades(self, dados):
        comodidades = [li.get_text(strip=True) for li in self.soup.select("div[class*='amenities'] li, ul[class*='feature'] li") if len(li.get_text(strip=True)) < 50]
        if comodidades:
            dados['comodidades_imovel'] = ", ".join(list(dict.fromkeys(comodidades)))

    def extrair_dados_do_anuncio(self):
        if not self.soup: return None
        dados = {'url_anuncio': self.url_atual}

        h1 = self.soup.find('h1')
        dados['titulo'] = h1.get_text(strip=True) if h1 else None

        self._extrair_endereco_e_bairro(dados)
        self._extrair_preco_e_custos(dados)
        self._extrair_caracteristicas(dados)
        self._extrair_detalhes_adicionais(dados)
        self._extrair_comodidades(dados)
        return dados

    def extrair_descricao_anuncio(self):
        if not self.soup: return 'Descrição não encontrada.'
        desc_tag = self.soup.select_one("p.description__content--text") or self.soup.select_one("[class*='description']")
        return desc_tag.get_text(separator='\n', strip=True) if desc_tag else 'Descrição não encontrada.'

    # =========================================================================
    # ETAPA 4: CHECKPOINT INCREMENTAL
    # =========================================================================

    def _salvar_lote(self, lote_dados):
        dados_totais = []
        if os.path.exists(self.arquivo_saida):
            with open(self.arquivo_saida, 'r', encoding='utf-8') as f:
                try: dados_totais = json.load(f)
                except json.JSONDecodeError: dados_totais = []

        dados_totais.extend(lote_dados)
        with open(self.arquivo_saida, 'w', encoding='utf-8') as f:
            json.dump(dados_totais, f, ensure_ascii=False, indent=4)

        print(f"  💾 Checkpoint salvo em '{self.arquivo_saida}' (Total acumulado: {len(dados_totais)} imóveis)")

    # =========================================================================
    # PIPELINE PRINCIPAL
    # =========================================================================

    def main(self):
        print("🚀 Iniciando pipeline - ZAP Imóveis (João Pessoa - Venda)\n")
        with sync_playwright() as p:
            navegador = p.chromium.launch(
                headless=True,
                args=['--disable-blink-features=AutomationControlled', '--no-sandbox', '--disable-setuid-sandbox']
            )
            contexto = navegador.new_context(
                user_agent=USER_AGENT,
                locale='pt-BR',
                viewport={'width': 1366, 'height': 768},
                extra_http_headers=EXTRA_HEADERS
            )
            self.pagina = contexto.new_page()

            links_finais = self.get_links_apartamentos_venda_jp()
            if not links_finais:
                print("❌ Nenhum link encontrado.")
                navegador.close()
                return

            print(f"\n⚙️ Extraindo dados de {len(links_finais)} imóveis...\n")
            lote_dados = []

            for i, link in enumerate(links_finais):
                print(f"[{i+1}/{len(links_finais)}] {link}")
                self.url_atual = link

                sucesso = self._carregar_html_da_pagina()
                if sucesso:
                    dados = self.extrair_dados_do_anuncio()
                    if dados:
                        descricao = self.extrair_descricao_anuncio()
                        dados['descricao_completa'] = descricao
                        lote_dados.append(dados)
                        print(f"  ✔ {dados.get('titulo', 'Sem título')[:65]}")

                if (i + 1) % self.tamanho_lote == 0 or (i + 1) == len(links_finais):
                    if lote_dados:
                        self._salvar_lote(lote_dados)
                        lote_dados = []

            navegador.close()
        print(f"\n🎉 Pipeline concluído com sucesso! Dados salvos em '{self.arquivo_saida}'.")

print("✅ Classe ScraperZapImoveis carregada com sucesso!")

✅ Classe ScraperZapImoveis carregada com sucesso!


## 🧪 Célula 4 — Teste: Coleta de Links (página 1)

In [4]:
def _worker_coleta_links():
    with sync_playwright() as p:
        navegador = p.chromium.launch(
            headless=True,
            args=['--disable-blink-features=AutomationControlled', '--no-sandbox', '--disable-setuid-sandbox']
        )
        contexto = navegador.new_context(
            user_agent=USER_AGENT,
            locale='pt-BR',
            viewport={'width': 1366, 'height': 768},
            extra_http_headers=EXTRA_HEADERS
        )
        pagina = contexto.new_page()

        print("Acessando página 1 de listagem...")
        pagina.goto("https://www.zapimoveis.com.br/venda/apartamentos/pb+joao-pessoa/?pagina=1", wait_until="domcontentloaded", timeout=45000)
        time.sleep(2)
        html = pagina.content()
        soup = BeautifulSoup(html, 'html.parser')

        links = []
        for a in soup.find_all('a', href=True):
            href = a['href']
            if '/imovel/' in href and 'joao-pessoa' in href.lower():
                url = href if href.startswith('http') else f"https://www.zapimoveis.com.br{href}"
                links.append(url)

        links_unicos = list(dict.fromkeys(links))
        navegador.close()
        return links_unicos

# Executa na thread isolada para prevenir erros no evento do Jupyter
links_teste = executar_em_thread(_worker_coleta_links)
print(f"\n✅ Links encontrados na página 1: {len(links_teste)}")
for link in links_teste[:5]:
    print(f"  → {link}")

NotImplementedError: 

## 🧪 Célula 5 — Teste: Extração de 1 Anúncio

In [ ]:
try:
    URL_TESTE = links_teste[0]
except (NameError, IndexError):
    URL_TESTE = "https://www.zapimoveis.com.br/imovel/venda-apartamento-3-quartos-com-piscina-jardim-oceania-joao-pessoa-pb-142m2-id-2870055299/?source=ranking%2Crp"

print(f"Testando extração em:\n  {URL_TESTE}\n")

def _worker_extracao(url):
    scraper = ScraperZapImoveis()
    with sync_playwright() as p:
        nav = p.chromium.launch(
            headless=True,
            args=['--disable-blink-features=AutomationControlled', '--no-sandbox', '--disable-setuid-sandbox']
        )
        ctx = nav.new_context(
            user_agent=USER_AGENT,
            locale='pt-BR',
            viewport={'width': 1366, 'height': 768},
            extra_http_headers=EXTRA_HEADERS
        )
        scraper.pagina = ctx.new_page()
        scraper.url_atual = url

        sucesso = scraper._carregar_html_da_pagina()
        if sucesso:
            dados = scraper.extrair_dados_do_anuncio()
            descricao = scraper.extrair_descricao_anuncio()
            dados['descricao_completa'] = descricao
            nav.close()
            return dados
        else:
            nav.close()
            return None

# Executa na thread isolada
dados_teste = executar_em_thread(_worker_extracao, URL_TESTE)

if dados_teste:
    print("📦 Dados extraídos com sucesso:")
    for chave, valor in dados_teste.items():
        if chave == 'descricao_completa':
            preview = str(valor)[:120].replace('\n', ' ')
            print(f"  {chave:25s}: {preview}...")
        else:
            print(f"  {chave:25s}: {valor}")
else:
    print("❌ Falha ao extrair dados do anúncio.")

## 🚀 Célula 6 — Pipeline Completo

> ⚠️ Executa a coleta completa de apartamentos à venda em João Pessoa - PB no ZAP Imóveis. Salva em lotes no arquivo `imoveis_joao_pessoa_zap.json`.

In [ ]:
def _worker_main():
    scraper = ScraperZapImoveis()
    scraper.main()

executar_em_thread(_worker_main)

## 📊 Célula 7 — Inspeção dos Dados Coletados

In [ ]:
import pandas as pd

if os.path.exists(ARQUIVO_SAIDA):
    with open(ARQUIVO_SAIDA, 'r', encoding='utf-8') as f:
        dados_carregados = json.load(f)

    df = pd.DataFrame(dados_carregados)
    print(f"📦 Total de registros coletados: {len(df)}")
    print(f"\n📋 Colunas ({len(df.columns)}):")
    for col in df.columns:
        preenchidos = df[col].notna().sum()
        pct = (preenchidos / len(df)) * 100
        print(f"  {col:30s} → {preenchidos:4d} ({pct:.1f}%)")
    
    display(df.head(3))
else:
    print(f"Arquivo '{ARQUIVO_SAIDA}' ainda não foi gerado. Execute a Célula 6 primeiro.")